In [2]:
import cv2
import mediapipe as mp
import numpy as np
import time
import tkinter as tk
from pynput.mouse import Controller, Button

mouse = Controller()

In [3]:
root = tk.Tk()
screen_w, screen_h = root.winfo_screenwidth(), root.winfo_screenheight()
root.destroy()

mp_hands = mp.solutions.hands
hands = mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=1,
    min_detection_confidence=0.7,
    min_tracking_confidence=0.7
)
mp_draw = mp.solutions.drawing_utils


I0000 00:00:1790967256.945554  104341 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1790967256.953703  104445 gl_context.cc:357] GL version: 3.2 (OpenGL ES 3.2 Mesa 26.2.2), renderer: AMD Radeon Graphics (radeonsi, renoir, ACO, DRM 3.64, 7.2.5-200.fc44.x86_64)


In [4]:
def get_landmark_list(hand_landmarks, frame_w, frame_h):
    lm_list = []
    for id, lm in enumerate(hand_landmarks.landmark):
        cx, cy = int(lm.x * frame_w), int(lm.y * frame_h)
        lm_list.append((id, cx, cy))
    return lm_list

In [5]:
def fingers_up(lm_list):

    fingers = []
    tips_ids = [4, 8, 12, 16, 20]

    # Thumb
    if lm_list[tips_ids[0]][1] > lm_list[tips_ids[0] - 1][1]:
        fingers.append(1)
    else:
        fingers.append(0)

    # Other 4 fingers (compare y-coordinates)
    for id in range(1, 5):
        if lm_list[tips_ids[id]][2] < lm_list[tips_ids[id] - 2][2]:
            fingers.append(1)
        else:
            fingers.append(0)

    return fingers

INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1790967256.981621  104431 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


In [6]:
smoothening = 10
prev_x, prev_y = 0, 0
curr_x, curr_y = 0, 0

frame_reduction = 100  
click_delay = 0.3
last_click_time = 0

In [7]:
cap = cv2.VideoCapture(0)
cap.set(3, 640)
cap.set(4, 480)

cv2.namedWindow("Virtual Mouse", cv2.WINDOW_NORMAL)
cv2.resizeWindow("Virtual Mouse", 1280, 720)

global prev_x, prev_y, curr_x, curr_y, last_click_time

while True:
    success, img = cap.read()
    if not success:
        break

    img = cv2.flip(img, 1)
    frame_h, frame_w, _ = img.shape
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    results = hands.process(img_rgb)
    
    if results.multi_hand_landmarks:
        hand_landmarks = results.multi_hand_landmarks[0]
        mp_draw.draw_landmarks(img, hand_landmarks, mp_hands.HAND_CONNECTIONS)
        cv2.circle(img, (10, 10), 10, (255, 0, 0), cv2.FILLED)
        lm_list = get_landmark_list(hand_landmarks, frame_w, frame_h)
    
        if len(lm_list) != 0:
            x1, y1 = lm_list[8][1], lm_list[8][2]   # index finger tip
            x2, y2 = lm_list[12][1], lm_list[12][2] # middle finger tip

            fingers = fingers_up(lm_list)

            # Only Index finger up = Move cursor
            if fingers[1] == 1 and fingers[2] == 0:
                x3 = np.interp(x1, (frame_reduction, frame_w - frame_reduction), (0, screen_w))
                y3 = np.interp(y1, (frame_reduction, frame_h - frame_reduction), (0, screen_h))

                curr_x = prev_x + (x3 - prev_x) / smoothening
                curr_y = prev_y + (y3 - prev_y) / smoothening

                mouse.position = (curr_x, curr_y)  
                prev_x, prev_y = curr_x, curr_y

            # Index + Middle finger up = Click
            if fingers[1] == 1 and fingers[2] == 1:
                distance = np.hypot(x2 - x1, y2 - y1)
                cv2.line(img, (x1, y1), (x2, y2), (0, 255, 0), 3)

                if distance < 40:  # fingers close together = click
                    current_time = time.time()
                    if current_time - last_click_time > click_delay:
                        mouse.click(Button.left, 1)
                        cv2.circle(img, ((x1+x2)//2, (y1+y2)//2), 15, (0, 255, 0), cv2.FILLED)
                        last_click_time = current_time

    cv2.imshow("Virtual Mouse", img)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

W0000 00:00:1790967257.054476  104430 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
QFontDatabase: Cannot find font directory /home/yasir/venvs/virtualmouse/lib64/python3.11/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/yasir/venvs/virtualmouse/lib64/python3.11/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/yasir/venvs/virtualmouse/lib64/python3.11/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/yasir/venvs/virtualmouse/lib64/python3.11/si

In [ ]:
cap.release()
cv2.destroyAllWindows()